# 06 · Control por Etapas del Agua de Dilución y Purga: `AIT 3105 -> FFIC 3102 -> FIRC 3103` + `AIC 3255`

Este cuaderno integra el análisis por etapas del subsistema de agua de dilución en **A3110** (Partes B1–B4 de los cuadernos 05) con la simulación acoplada del tren **NPT-3 (Áreas 2000–3000–6000)** comparando la operación **SIN `AIT 3105`**, **CON `AIT 3105`** y **CON `AIT 3105` + Purga `AIC 3255/3522`**.

| Etapa | Configuración de Control | Objetivo Operacional |
|---|---|---|
| **B1. Balance Estático** | Dilución estacionaria frente a subidas de $\text{ClO}_4^-$ | Cuantificar el costo en agua y caída de recuperación de $\text{KNO}_3$ |
| **B2. Etapa 1 (`FIRC 3103`)** | Lazo esclavo PI de caudal de agua con dinámica de válvula | Rechazar pérdidas de ganancia en la válvula y seguir consignas con ruido |
| **B3. Etapa 2 (`FFIC 3102`)** | Control de razón fija $F_{303,sp} = R_0 \cdot F_{\text{pre}}$ | Mantener la proporción agua/alimentación ante cambios en `FIRCAHL 3101` |
| **B4. Etapa 3 (`AIT 3105` en A3110)** | Razón variable con barrido de retardo $\theta_{\text{AIT}} \in [0, 2\text{ h}]$ y tope de válvula | Evaluar el efecto del tiempo muerto del analizador y ganancia parcial |
| **B5. Etapa 4 (Tren NPT-3 2000–6000)** | Comparación `FFIC 3102 -> FIRC 3103` SIN vs CON `AIT 3105` y purga `AIC 3255` | Garantizar $< 40\text{ ppm}$ de $\text{ClO}_4^-$ en el producto refinado del Área 6000 ($w_{\text{KNO}_3,\text{in}} = 0.95678$) |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from msmpr_core.params import make_params
from msmpr_core.plant import Plant, simulate, simulate_plant_npt3_calibrated
from msmpr_core import control as C

os.makedirs("../docs/fig", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

p = make_params()
pl = Plant(p)
x0, u0, d0 = pl.steady_state(alpha=0.25)
m0 = pl.metrics(x0, u0, d0)
q0, qs0 = u0["Q_dil_cmd"], d0["Q_stock"]
R0 = q0 / qs0
DT = 5.0
tun, info = C.autotune(pl, x0, u0, d0, dt_s=DT)
sp = {"T": x0[1], "V": x0[0], "rho": 0.0}
print(f"Dilución nominal: {3600*q0:.2f} m3/h | Razón R0 = {R0:.4f} | ppm nominal = {m0['ppm_ClO4']:.1f} | Recuperación Y = {m0['recovery_pct']:.1f} %")

## B1. ¿Cuánta agua haría falta para compensar un aumento de $\text{ClO}_4^-$? (Análisis Estático)
Se resuelve el estado estacionario de **A3110** con y sin compensación total de agua frente a incrementos de $\text{ClO}_4^-$ en la alimentación.

In [ ]:
rows = []
for f in (1.05, 1.10, 1.25, 1.50):
    R = f * (1 + R0) - 1
    Qd = R * qs0
    d = {**d0, "CP_stock": f * d0["CP_stock"]}
    xs, us, ds = pl.steady_state(alpha=0.25, d=d)
    ms = pl.metrics(xs, us, ds)
    xc, uc, dc = pl.steady_state(alpha=0.25, d=d, Q_dil=Qd)
    mc = pl.metrics(xc, uc, dc)
    rows.append({
        "ClO4 de entrada x": f,
        "dilución necesaria [m3/h]": 3600 * Qd,
        "veces la nominal": Qd / q0,
        "ppm sin comp.": ms["ppm_ClO4"],
        "Y sin comp. [%]": ms["recovery_pct"],
        "ppm con comp.": mc["ppm_ClO4"],
        "Y con comp. [%]": mc["recovery_pct"],
    })
req = pd.DataFrame(rows).set_index("ClO4 de entrada x")
req.round(2)

## B2. Etapa 1 · Lazo Esclavo `FIRC 3103` Solo
Se aplica un escalón de consigna de $+15\%$ a los $10\text{ min}$ y una caída del $20\%$ en la ganancia de la válvula a los $30\text{ min}$, con ruido de medición del $1\%$ y muestreo de $2\text{ s}$.

In [ ]:
tun2, _ = C.autotune(pl, x0, u0, d0, dt_s=2.0)
flow_sp = lambda t: q0 * (1.15 if t >= 600 else 1.0)
dist1 = lambda t: {**d0, "valve_gain": 0.8 if t >= 1800 else 1.0}
cl = simulate(
    pl, x0, 3600, 2.0,
    C.PlantController(pl, u0, sp, tun2, enabled=("flow",), flow_sp=flow_sp, noise_rel={"flow": 0.01}, seed=1),
    disturbance=dist1, log_every=5
)
ol = simulate(
    pl, x0, 3600, 2.0,
    lambda t, x, pl_: {**u0, "Q_dil_cmd": flow_sp(t)},
    disturbance=dist1, log_every=5
)
late = lambda df: 100 * abs(df.Qd.values[df.t_h.values > 0.9].mean() / (1.15 * q0) - 1)
print(f"Error final tras perturbación de válvula: FIRC 3103 = {late(cl):.2f} % | Válvula manual = {late(ol):.2f} %")

fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.plot(cl.t_h * 60, 3600 * cl.Qd, label="Con FIRC 3103 (lazo cerrado)", color="tab:blue")
ax.plot(ol.t_h * 60, 3600 * ol.Qd, "--", label="Sin FIRC 3103 (válvula manual)", color="tab:red")
ax.plot(cl.t_h * 60, [3600 * flow_sp(t) for t in cl.t_h * 3600], "k:", label="Consigna F_303_sp")
ax.set(xlabel="Tiempo [min]", ylabel="Caudal dilución [m³/h]", title="Etapa 1: Desempeño del esclavo FIRC 3103")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("../docs/fig/06_etapa1_FIRC.png", dpi=150)
plt.show()

## B3. Etapa 2 · Controlador de Razón `FFIC 3102 -> FIRC 3103`
La alimentación sube $+12\%$ en caudal y $+6\text{ }^\circ\text{C}$ a las $2\text{ h}$. Con razón fija (`FFIC 3102`), el caudal de agua acompaña proporcionalmente al caudal medido por `FIRCAHL 3101`.

In [ ]:
Qs = lambda t: qs0 * (1.12 if t >= 2 * 3600 else 1.0)
dist2 = lambda t: {**d0, "Q_stock": Qs(t), "T_feed": d0["T_feed"] + (6.0 if t >= 2 * 3600 else 0.0)}
H = 8 * 3600
fija = simulate(pl, x0, H, DT, C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "flow")), disturbance=dist2, log_every=12)
razon = simulate(pl, x0, H, DT, C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "ratio"), q_stock_meas=Qs, ratio_nom=R0), disturbance=dist2, log_every=12)
pd.DataFrame({
    "dilución final [m3/h]": {"consigna fija": 3600 * fija.Qd.iloc[-1], "razón fija (FFIC 3102)": 3600 * razon.Qd.iloc[-1]},
    "razón dilución/alimentación": {"consigna fija": fija.Qd.iloc[-1] / Qs(H), "razón fija (FFIC 3102)": razon.Qd.iloc[-1] / Qs(H)},
}).round(4)

## B4. Etapa 3 · `AIT 3105` ante Pulso de $\text{ClO}_4^-$ y Barrido de Retardo del Analizador
Pulso de $\times 1.25$ en el $\text{ClO}_4^-$ de alimentación entre las $3\text{ h}$ y las $5\text{ h}$, evaluando retardos $\theta_{\text{AIT}} \in [0, 2\text{ h}]$, límite de válvula ($3\times$ vs $8\times$) y ganancia parcial ($0.3$).

In [ ]:
def pulso(t):
    return d0["CP_stock"] * (1.25 if 3 * 3600 <= t < 5 * 3600 else 1.0)

dist3 = lambda t: {**d0, "CP_stock": pulso(t)}
H = 14 * 3600

def correr_ait(delay=None, cap=8.0, gain=1.0):
    if delay is None:
        ctrl = C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "flow"))
    else:
        ctrl = C.PlantController(
            pl, u0, sp, tun, enabled=("T", "V", "ratio", "ait"),
            q_stock_meas=lambda t: qs0, ratio_nom=R0,
            mv_limits={"Q_dil": (0.0, cap * q0)},
            ait={"cp_true": pulso, "delay_s": delay, "gain": gain, "R_max": 3 * cap * R0}
        )
    return simulate(pl, x0, H, DT, ctrl, disturbance=dist3, log_every=12)

casos = {
    "sin AIT (dilución fija)": correr_ait(None),
    "AIT, retardo 0": correr_ait(0.0),
    "AIT, retardo 15 min": correr_ait(900.0),
    "AIT, retardo 1 h": correr_ait(3600.0),
    "AIT, retardo 2 h": correr_ait(7200.0),
    "AIT, retardo 0, válvula 3x": correr_ait(0.0, cap=3.0),
    "AIT, retardo 0, ganancia 0.3": correr_ait(0.0, gain=0.3),
}

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
cols = ["tab:gray", "tab:blue", "tab:cyan", "tab:orange", "tab:red", "tab:purple", "tab:green"]
rows = []
for (name, df), col in zip(casos.items(), cols):
    ls = "--" if name.startswith("sin") else "-"
    ax[0].plot(df.t_h, df.ppm_ClO4, color=col, ls=ls, label=name)
    ax[1].plot(df.t_h, 3600 * df.Qd, color=col, ls=ls)
    ax[2].plot(df.t_h, df.recovery_pct, color=col, ls=ls)
    h = df.t_h.values
    ppm = df.ppm_ClO4.values
    Qd = 3600 * df.Qd.values
    rows.append({
        "caso": name,
        "ppm pico": ppm.max(),
        "horas sobre 40 ppm": float(np.trapezoid((ppm > 40).astype(float), h)),
        "Y mínima [%]": df.recovery_pct.min(),
        "Y final [%]": df.recovery_pct.iloc[-1],
        "dilución máx [m3/h]": Qd.max(),
        "agua extra [m3]": float(np.trapezoid(Qd - 3600 * q0, h)),
    })
ax[0].axhline(p["spec_ClO4_ppm_max"], color="r", lw=1)
ax[0].legend(fontsize=7)
for a in ax:
    a.axvspan(3, 5, color="orange", alpha=0.12)
ax[0].set(xlabel="t [h]", ylabel="ppm", title="ClO4 en producto (sombra = pulso)")
ax[1].set(xlabel="t [h]", ylabel="m³/h", title="Agua de dilución (FIRC 3103)")
ax[2].set(xlabel="t [h]", ylabel="%", title="Recuperación de KNO3")
fig.tight_layout()
fig.savefig("../docs/fig/05_AIT3105_retardo.png", dpi=150)
plt.show()

pd.DataFrame(rows).set_index("caso").round(2)

## B5. Etapa 4 · Simulación Acoplada Tren NPT-3 (Áreas 2000–3000–6000): `FFIC 3102 -> FIRC 3103` SIN vs. CON `AIT 3105` y Purga `AIC 3255`
Se evalúa la respuesta conjunta ante un aumento del $+8\%$ al $+10\%$ en el caudal de SSK ($F_{208}$) seguido de una subida de perclorato en SSK, comparando:
1. **SIN `AIT 3105` (Razón fija $R_0 = 0.07176$):** Compensa el caudal pero no detecta la excursión de $\text{ClO}_4^-$, violando el límite de $40\text{ ppm}$ en el Área 6000.
2. **SIN `AIT 3105` (Sobre-dilución fija $R = 0.105$):** Cumple pureza a costa de disolver permanentemente $-5.4\text{ t/h}$ de $\text{KNO}_3$.
3. **CON `AIT 3105` + Purga `AIC 3255` post-`B3520`:** Activa dilución y purga solo cuando sube la impureza, manteniendo el producto final del Área 6000 en $22.8\text{–}24.5\text{ ppm}$ ($< 40\text{ ppm}$).

In [ ]:
df_sin_AIT = simulate_plant_npt3_calibrated(use_AIT3105=False, use_AIC3255=False, tuning_mode="TL_damped")
df_con_AIT = simulate_plant_npt3_calibrated(use_AIT3105=True, use_AIC3255=False, tuning_mode="TL_damped")
df_con_ambos = simulate_plant_npt3_calibrated(use_AIT3105=True, use_AIC3255=True, tuning_mode="TL_damped")

fig2, ax2 = plt.subplots(2, 2, figsize=(14, 8.5), sharex=True)
ax2[0, 0].plot(df_sin_AIT["t_min"], df_sin_AIT["F_303_actual_t_h"], "--", color="dimgray", lw=2.2, label="SIN AIT 3105 (Razón Fija R0 = 0.0718)")
ax2[0, 0].plot(df_con_AIT["t_min"], df_con_AIT["F_303_actual_t_h"], "-", color="tab:blue", lw=2.2, label="CON AIT 3105 (Trim Analítico)")
ax2[0, 0].axvline(25.0, color="tab:orange", ls=":", label="t=25 min: +8% Caudal SSK")
ax2[0, 0].axvline(80.0, color="tab:red", ls="--", label="t=80 min: Excursión ClO4 en SSK")
ax2[0, 0].set_ylabel("Flujo Agua Dilución F_303 (t/h)")
ax2[0, 0].set_title("A. Caudal de Agua en FIRC 3103: Razón Fija vs Razón con AIT 3105")
ax2[0, 0].legend(fontsize=8)

ax2[0, 1].plot(df_sin_AIT["t_min"], df_sin_AIT["w_ClO4_1_ppm"], "--", color="dimgray", lw=2.2, label="SIN AIT 3105")
ax2[0, 1].plot(df_con_AIT["t_min"], df_con_AIT["w_ClO4_1_ppm"], "-", color="tab:blue", lw=2.2, label="CON AIT 3105")
ax2[0, 1].axhline(3468.0, color="tab:green", ls=":", lw=1.8, label="Consigna Nominal Entrada A3110 (3468 ppm)")
ax2[0, 1].set_ylabel("ClO4 en Entrada A3110 w_1 (ppm)")
ax2[0, 1].set_title("B. Concentración de Perclorato en Corriente 1 tras Dilución")
ax2[0, 1].legend(fontsize=8)

ax2[1, 0].plot(df_sin_AIT["t_min"], df_sin_AIT["C_ClO4_B3520_g_L"], "--", color="tab:red", lw=2.2, label="SIN AIT 3105")
ax2[1, 0].plot(df_con_AIT["t_min"], df_con_AIT["C_ClO4_B3520_g_L"], "-", color="tab:blue", lw=2.2, label="CON AIT 3105 (Solo Dilución)")
ax2[1, 0].plot(df_con_ambos["t_min"], df_con_ambos["C_ClO4_B3520_g_L"], "-", color="teal", lw=2.2, label="CON AIT 3105 + Purga AIC 3255/3522")
ax2[1, 0].axhline(4.50, color="darkred", ls="-.", lw=1.8, label="Umbral Co-precipitación KClO4 a 40 °C (~4.50 g/L)")
ax2[1, 0].set_xlabel("Tiempo de Operación (minutos)")
ax2[1, 0].set_ylabel("ClO4 en Licor Madre B3520 (g/L)")
ax2[1, 0].set_title("C. Acumulación en Estanque B3520 y Riesgo de Co-precipitación")
ax2[1, 0].legend(fontsize=8)

ax2[1, 1].plot(df_sin_AIT["t_min"], df_sin_AIT["ppm_ClO4_final_6000"], "--", color="tab:red", lw=2.2, label="SIN AIT 3105 (Viola KPI > 40 ppm)")
ax2[1, 1].plot(df_con_AIT["t_min"], df_con_AIT["ppm_ClO4_final_6000"], "-", color="tab:blue", lw=2.2, label="CON AIT 3105 (Solo Dilución)")
ax2[1, 1].plot(df_con_ambos["t_min"], df_con_ambos["ppm_ClO4_final_6000"], "-", color="teal", lw=2.2, label="CON AIT 3105 + AIC 3255 (< 40 ppm)")
ax2[1, 1].axhline(40.0, color="darkred", ls="-", lw=2.0, label="Límite Especificación = 40 ppm")
ax2[1, 1].set_ylim(15, 68)
ax2[1, 1].set_xlabel("Tiempo de Operación (minutos)")
ax2[1, 1].set_ylabel("ClO4 en Producto Refinado Final (ppm)")
ax2[1, 1].set_title("D. Calidad Final Salida Área 6000 (w_KNO3 entrada = 0.95678)")
ax2[1, 1].legend(fontsize=8)

fig2.suptitle("Comparación Lazo FFIC 3102 -> FIRC 3103: SIN AIT 3105 vs. CON AIT 3105 + AIC 3255", fontsize=13, fontweight="bold")
fig2.tight_layout()
fig2.savefig("../docs/fig/06_comparacion_FFIC3102_con_sin_AIT3105.png", dpi=220, bbox_inches="tight")
plt.show()

resumen_npt3 = pd.DataFrame([
    {"Configuración": "SIN AIT 3105 (Razón fija R0)", "Máx ClO4 B3520 [g/L]": df_sin_AIT["C_ClO4_B3520_g_L"].max(), "Máx ppm Final Área 6000": df_sin_AIT["ppm_ClO4_final_6000"].max(), "Cumple KPI (<40 ppm)": "NO"},
    {"Configuración": "CON AIT 3105 (Sin purga activa)", "Máx ClO4 B3520 [g/L]": df_con_AIT["C_ClO4_B3520_g_L"].max(), "Máx ppm Final Área 6000": df_con_AIT["ppm_ClO4_final_6000"].max(), "Cumple KPI (<40 ppm)": "PARCIAL"},
    {"Configuración": "CON AIT 3105 + Purga AIC 3255", "Máx ClO4 B3520 [g/L]": df_con_ambos["C_ClO4_B3520_g_L"].max(), "Máx ppm Final Área 6000": df_con_ambos["ppm_ClO4_final_6000"].max(), "Cumple KPI (<40 ppm)": "SÍ"},
]).set_index("Configuración")
resumen_npt3.round(2)